In [ ]:
# Install required packages
!pip install paho-mqtt tensorflow numpy pandas h5py scikit-learn scipy

# Clear any existing TensorFlow sessions to avoid duplicate registrations
import tensorflow as tf
tf.keras.backend.clear_session()

# Import necessary libraries
import numpy as np
import pandas as pd
import h5py
from sklearn.model_selection import train_test_split
from scipy import fft
from tensorflow.keras.optimizers import Adam
import keras
from tensorflow.keras.layers import LSTM, Dense, Dropout, TimeDistributed, Input
import paho.mqtt.client as mqtt
import json
import time
import random
import os

In [ ]:
# Manual dataset setup (replace with your dataset path)
DATASET_PATH = "/kaggle/input/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5"  # Update this path

# Check if dataset exists
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}. Please provide the correct path.")

# Load dataset
dataset_file = h5py.File(DATASET_PATH, "r")

# Define modulation classes
base_modulation_classes = [
    'OOK', '4ASK', '8ASK', 'BPSK', 'QPSK', '8PSK', '16PSK', '32PSK',
    '16APSK', '32APSK', '64APSK', '128APSK', '16QAM', '32QAM', '64QAM',
    '128QAM', '256QAM', 'AM-SSB-WC', 'AM-SSB-SC', 'AM-DSB-WC', 'AM-DSB-SC',
    'FM', 'GMSK', 'OQPSK'
]
selected_modulation_classes = ['4ASK', 'BPSK', 'QPSK', '16PSK', '16QAM', 'FM', 'AM-DSB-WC', '32APSK']
selected_classes_id = [base_modulation_classes.index(cls) for cls in selected_modulation_classes]
N_SNR = 4

# Load and prepare data
X_data = None
y_data = None
for id in selected_classes_id:
    start_idx = 106496 * (id + 1) - 4096 * N_SNR
    end_idx = 106496 * (id + 1)
    X_slice = dataset_file['X'][start_idx:end_idx]
    y_slice = dataset_file['Y'][start_idx:end_idx]
    if X_data is not None:
        X_data = np.concatenate([X_data, X_slice], axis=0)
        y_data = np.concatenate([y_data, y_slice], axis=0)
    else:
        X_data = X_slice
        y_data = y_slice

# Reshape for RNN and prepare labels
X_data = X_data.reshape(len(X_data), 32, 64)
y_data_df = pd.DataFrame(y_data)
for column in y_data_df.columns:
    if y_data_df[column].sum() == 0:
        y_data_df = y_data_df.drop(columns=[column])
y_data_df.columns = selected_modulation_classes

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data_df, test_size=0.2, random_state=42, stratify=y_data_df.values.argmax(axis=1)
)

In [ ]:

# Visualize one signal from each modulation class in continuous time domain
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(selected_modulation_classes), 1, figsize=(12, 35))
for i, mod in enumerate(selected_modulation_classes):
    mod_idx = y_data_df[mod] == 1
    signal = X_data[mod_idx][0]
    I = signal[0]
    Q = signal[1]
    t = np.arange(len(I))
    axes[i].plot(t, I, label='I', alpha=0.7)
    axes[i].plot(t, Q, label='Q', alpha=0.7)
    axes[i].set_title(f"Modulation: {mod}")
    axes[i].legend()
    axes[i].grid(True)

In [ ]:
# Spectrum sensing functions
def compute_psd(signal):
    fft_result = fft.fft(signal, axis=-1)
    psd = np.abs(fft_result) ** 2 / signal.shape[-1]
    return psd

def analyze_spectrum(X_data):
    psd_data = compute_psd(X_data)
    psd_mean = np.mean(psd_data, axis=(0, 1))
    threshold = np.percentile(psd_mean, 30)
    occupied_channels = psd_mean > threshold
    return psd_data, occupied_channels

In [ ]:
# Normalize signal
def normalize_signal(signal):
    mean = np.mean(signal, axis=(1, 2), keepdims=True)
    std = np.std(signal, axis=(1, 2), keepdims=True)
    return (signal - mean) / (std + 1e-8)

X_train = normalize_signal(X_train)
X_test = normalize_signal(X_test)

# Perform spectrum analysis on training data
psd_train, occupied_channels = analyze_spectrum(X_train)

In [ ]:
# Define RNN model for classification
def create_model():
    input_layer = Input(shape=(32, 64))
    x = LSTM(128, return_sequences=True)(input_layer)
    x = Dropout(0.2)(x)
    x = LSTM(64)(x)
    x = Dropout(0.2)(x)
    x = Dense(128, activation='relu')(x)
    output = Dense(len(selected_modulation_classes), activation='softmax')(x)
    model = keras.Model(inputs=input_layer, outputs=output)
    model.compile(loss='categorical_crossentropy', 
                  optimizer=Adam(learning_rate=0.0001), 
                  metrics=['accuracy'])
    return model

model = create_model()

In [ ]:
# Dynamic Spectrum Allocation
class SpectrumManager:
    def __init__(self, n_channels=64):
        self.n_channels = n_channels
        self.channel_status = np.zeros(n_channels, dtype=bool)
        self.channel_quality = np.ones(n_channels)
        self.last_allocation = time.time()

    def update_status(self, occupied_channels):
        self.channel_status = occupied_channels
        self.channel_quality[self.channel_status] *= 0.9
        self.channel_quality[~self.channel_status] = np.minimum(
            self.channel_quality[~self.channel_status] + 0.1, 1.0
        )

    def allocate_channel(self):
        free_channels = np.where(~self.channel_status)[0]
        if len(free_channels) == 0:
            return None
        qualities = self.channel_quality[free_channels]
        best_channel = free_channels[np.argmax(qualities)]
        self.channel_status[best_channel] = True
        self.last_allocation = time.time()
        return best_channel

    def get_allocation_stats(self):
        return {
            "occupied_channels": int(np.sum(self.channel_status)),
            "free_channels": int(self.n_channels - np.sum(self.channel_status)),
            "average_quality": float(np.mean(self.channel_quality))
        }

In [ ]:
# IoT Node class for MQTT publishing
class IoTNode:
    def __init__(self, broker="localhost", port=1883, topic="modulation/class", spectrum_manager=None):
        self.client = mqtt.Client(callback_api_version=mqtt.CallbackAPIVersion.VERSION2)
        self.broker = broker
        self.port = port
        self.topic = topic
        self.spectrum_manager = spectrum_manager
        self.client.on_connect = self.on_connect
        self.client.connect(broker, port)
        self.client.loop_start()

    def on_connect(self, client, userdata, flags, reason_code, properties):
        print(f"Publisher connected to {self.broker} with code {reason_code}")

    def publish(self, message):
        payload = json.dumps(message)
        self.client.publish(self.topic, payload)
        print(f"Sent: {payload}")

    def stop(self):
        self.client.loop_stop()
        self.client.disconnect()

In [ ]:
# Train with IoT updates
spectrum_manager = SpectrumManager(n_channels=64)
iot_node = IoTNode(broker="test.mosquitto.org", topic="modulation/train", spectrum_manager=spectrum_manager)

class IoTCallback(tf.keras.callbacks.Callback):
    def __init__(self, iot_node):
        super().__init__()
        self.iot_node = iot_node
        self.best_val_accuracy = 0.0
        self.patience = 5  # Increased to 5 epochs to wait for improvement
        self.no_improvement_count = 0

    def on_epoch_end(self, epoch, logs=None):
        _, occupied_channels = analyze_spectrum(X_train)
        self.iot_node.spectrum_manager.update_status(occupied_channels)
        allocated_channel = self.iot_node.spectrum_manager.allocate_channel()
        stats = self.iot_node.spectrum_manager.get_allocation_stats()
        message = {
            "epoch": int(epoch + 1),
            "accuracy": float(logs.get("accuracy", 0)),
            "val_accuracy": float(logs.get("val_accuracy", 0)),
            "allocated_channel": int(allocated_channel) if allocated_channel is not None else -1,
            "spectrum_stats": {
                "occupied_channels": int(stats["occupied_channels"]),
                "free_channels": int(stats["free_channels"]),
                "average_quality": float(stats["average_quality"])
            }
        }
        self.iot_node.publish(message)

        # Early stopping logic with relaxed threshold
        current_val_accuracy = logs.get("val_accuracy", 0)
        if current_val_accuracy > self.best_val_accuracy + 0.005:  # Lowered threshold to 0.5%
            self.best_val_accuracy = current_val_accuracy
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1
            if self.no_improvement_count >= self.patience:
                print(f"Stopping training: No improvement in validation accuracy for {self.patience} epochs.")
                self.model.stop_training = True

# Train the model with automated epochs
history = model.fit(
    X_train, y_train,
    batch_size=64,
    epochs=50,  # Max epochs; will stop earlier if no improvement
    validation_data=(X_test, y_test),
    callbacks=[IoTCallback(iot_node)]
)
iot_node.stop()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Check if model and test data are available (placeholders if not previously defined)
try:
    # Confusion Matrix
    cm = confusion_matrix(y_true_labels, y_pred_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

    # Plot Confusion Matrix
    
    fig, ax = plt.subplots(figsize=(10, 8))
    disp.plot(ax=ax, xticks_rotation=45, cmap='Blues')
    plt.title("Confusion Matrix")
    plt.grid(False)
    plt.tight_layout()
    plt.show()

except Exception as e:
    str(e)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_true_labels, y_pred_labels)
precision = precision_score(y_true_labels, y_pred_labels, average='macro', zero_division=0)
recall = recall_score(y_true_labels, y_pred_labels, average='macro', zero_division=0)
f1 = f1_score(y_true_labels, y_pred_labels, average='macro', zero_division=0)

print("Classification Accuracy:", accuracy)
print("Precision (Macro):", precision)
print("Recall (Macro):", recall)
print("F1 Score (Macro):", f1)

In [ ]:
# Real-time simulation (without anomaly detection)
def simulate_real_time_iot(model, X_test, y_test):
    spectrum_manager = SpectrumManager(n_channels=64)
    iot_node = IoTNode(broker="test.mosquitto.org", topic="modulation/predict", spectrum_manager=spectrum_manager)
    print("Starting real-time simulation. Press Ctrl+C to stop.")
    try:
        while True:
            idx = random.randint(0, len(X_test) - 1)
            sample = X_test[idx:idx+1]
            sample_y = y_test.iloc[idx:idx+1]
            psd_sample, occupied_channels = analyze_spectrum(sample)
            spectrum_manager.update_status(occupied_channels)
            allocated_channel = spectrum_manager.allocate_channel()
            stats = spectrum_manager.get_allocation_stats()
            pred = model.predict(sample, verbose=0)
            pred_class = selected_modulation_classes[np.argmax(pred[0])]
            true_class = selected_modulation_classes[np.argmax(sample_y.values[0])]
            confidence = float(np.max(pred[0]))
            message = {
                "predicted": pred_class,
                "true": true_class,
                "confidence": float(confidence),  # Ensure float
                "allocated_channel": int(allocated_channel) if allocated_channel is not None else -1,  # Convert to int
                "spectrum_stats": {
                    "occupied_channels": int(stats["occupied_channels"]),  # Convert to int
                    "free_channels": int(stats["free_channels"]),  # Convert to int
                    "average_quality": float(stats["average_quality"])  # Ensure float
                },
                "time": float(time.time())  # Ensure float
            }
            iot_node.publish(message)
            time.sleep(0.1)
    except KeyboardInterrupt:
        print("Stopping real-time simulation.")
        iot_node.stop()

# Run the simulation
simulate_real_time_iot(model, X_test, y_test)